## CONSTRUCTION DE LA TABLE DE FAIT

In [67]:
# importation des bibliotheques
import pandas as pd
from pathlib import Path

In [68]:
# chemin vers les datasets
data_dir=Path("../data/raw")

In [69]:
# chargement des fichiers csv
commande=pd.read_csv(data_dir/"olist_orders_dataset.csv")
articles=pd.read_csv(data_dir/"olist_order_items_dataset.csv")
reviews=pd.read_csv(data_dir/"olist_order_reviews_dataset.csv")
customers=pd.read_csv(data_dir/"olist_customers_dataset.csv")
vendeur=pd.read_csv(data_dir/"olist_sellers_dataset.csv")

In [70]:
print("nombre d'observation de chaque dataframe:")
print("total observation des commandes :", len(commande))
print("total observation des articles :", len(articles))
print("total observation des reviews :", len(reviews))
print("total observation des customers :", len(customers))
print("total observation des vendeurs :", len(vendeur))

nombre d'observation de chaque dataframe:
total observation des commandes : 99441
total observation des articles : 112650
total observation des reviews : 99224
total observation des customers : 99441
total observation des vendeurs : 3095


In [71]:
# conversion des dates en datetime
commande["order_purchase_timestamp"]=pd.to_datetime(commande["order_purchase_timestamp"])
commande["order_delivered_customer_date"]=pd.to_datetime(commande["order_delivered_customer_date"])
commande["order_estimated_delivery_date"]=pd.to_datetime(commande["order_estimated_delivery_date"])
reviews["review_answer_timestamp"]=pd.to_datetime(reviews["review_answer_timestamp"])
print("dates converties en datetime")

dates converties en datetime


In [72]:
# jointure entre commande et article
faits=articles.merge(
    commande[["order_id", "customer_id","order_status","order_purchase_timestamp","order_delivered_customer_date","order_estimated_delivery_date"]],
    on="order_id",
    how="left"
)
print("total observation apres fusion de fait et commande:", len(faits))
# verification de l'assersion
assert len(faits)==112650, f" resultat attendu: 112650, resultat obtenu: {len(faits)}"

total observation apres fusion de fait et commande: 112650


In [73]:
# calcul des delais en jours

# delai de livraison
faits["delai_livraison_jour"]=(
    faits["order_delivered_customer_date"]- faits["order_purchase_timestamp"]
).dt.days

# delai estime
faits["delai_estime_jour"]=(
    faits["order_estimated_delivery_date"]-faits["order_purchase_timestamp"]
).dt.days

# retard de livraison
faits["retard_jour"]=(
    faits["order_delivered_customer_date"]-faits["order_estimated_delivery_date"]
).dt.days

# verification du retard
faits["est_en_retard"]=faits["retard_jour"]>0

In [74]:
# afficahge des delais calcules
print("resultats des delais calcules:")
print("delais de livraison moyen: ", faits["delai_livraison_jour"].mean().round(3),"jours")
print("delai estime moyen en jour:", faits["delai_estime_jour"].mean().round(3),"jours")
print("retard moyen en jour: ", faits["retard_jour"].mean().round(3),"jours")

resultats des delais calcules:
delais de livraison moyen:  12.008 jours
delai estime moyen en jour: 23.47 jours
retard moyen en jour:  -12.03 jours


In [75]:
# dedloublonnage des avis pour ne garder que l'avis le plus recent
reviews_clean=(
    reviews.sort_values(["order_id","review_answer_timestamp","review_id"]).drop_duplicates("order_id", keep="last")
)

In [76]:
# verificaton des resultats
print("avis avant le dedoublonnage:", len(reviews))
print("avis apres dedounlonnage : ", len(reviews_clean))
assert len(reviews_clean)==98673, f"resultat attendu: 98673, resultat obtenu: {len(reviews_clean)}"

avis avant le dedoublonnage: 99224
avis apres dedounlonnage :  98673


In [77]:
# jointure sur les avis : left join pour conserver les commandes sans avis
faits=faits.merge(
    reviews_clean[["order_id","review_score"]],
    on="order_id",
    how="left"
)
print("totale observation apres fusion des avis:", len(faits))
assert len(faits)==112650, f"resultat attendu: 112650, resultat obtenu: {len(faits)}"

# nombre de reviews score manquant
total_review_na=faits["review_score"].isna().sum()
print("total observation des reviews manquant :", total_review_na)


totale observation apres fusion des avis: 112650
total observation des reviews manquant : 942


In [78]:
# jointure des regions(clients)
client=customers[["customer_id","customer_state"]].rename(
    columns={"customer_state":"region_client_etat"}
)
faits=faits.merge(client,on="customer_id",how="left")

print("totale observation apres fusion des clients:", len(faits))
assert len(faits)==112650
# verification
print("totale region_client_etat unique :", faits["region_client_etat"].nunique())

totale observation apres fusion des clients: 112650
totale region_client_etat unique : 27


In [79]:
# jointure sur region(vendeurs)
vendeur_clean=vendeur[["seller_id","seller_state"]].rename(columns={"seller_state":"region_vendeur_etat"})
faits=faits.merge(vendeur_clean,on="seller_id",how="left")

print("total observation apres jointure :", len(faits))
assert len(faits)==112650
print("total: region_vendeur unique :", faits["region_vendeur_etat"].nunique())

total observation apres jointure : 112650
total: region_vendeur unique : 23


In [80]:
# selection des colonnes suivantes
faits=faits.rename(columns={
    "order_id":"order_id",
    "order_item_id":"order_item_id",
    "customer_id":"client_id",
    "product_id":"produit_id",
    "seller_id":"vendeur_id",
    "order_status":"statut_commande",
    "order_purchase_timestamp":"date_achat",
    "price":"price",
    "freight_value":"freight_value",
    "delai_livraison_jour":"delai_livraison_jour",
    "delai_estime_jour":"delai_estime_jour",
    "retard_jour":"retard_jour",
    "est_en_retard":"est_en_retard",
    "review_score":"review_score",
    "region_client_etat":"region_client_etat",
    "region_vendeur_etat":"region_vendeur_etat"
})

# selection des colonnes finales
colonnes_finales=[
    "order_id","order_item_id","client_id","produit_id","vendeur_id","region_client_etat","region_vendeur_etat","date_achat","price","freight_value","delai_livraison_jour","delai_estime_jour","retard_jour","est_en_retard","review_score","statut_commande"
]

faits=faits[colonnes_finales]

print("colonne finale :", list(faits.columns))
print("total observation :", len(faits))

colonne finale : ['order_id', 'order_item_id', 'client_id', 'produit_id', 'vendeur_id', 'region_client_etat', 'region_vendeur_etat', 'date_achat', 'price', 'freight_value', 'delai_livraison_jour', 'delai_estime_jour', 'retard_jour', 'est_en_retard', 'review_score', 'statut_commande']
total observation : 112650


In [82]:
# verification finale

# nombre de ligne
assert len(faits)==112650 , f"resultat attendu: 112650, resultat obtenu: {len(faits)}"

# abscence de doublon sur order_id, order_item_id
total_doublon=faits.duplicated(subset=["order_id","order_item_id"]).sum()
assert total_doublon==0, f"resultat attendu: 0, resultat obtenu: {total_doublon}"
print("aucun doublon sur order_id et order_items_id")

# abscence des NULL sur les colonnes obligatoires
for col in ["order_id","order_item_id","client_id","produit_id","vendeur_id","date_achat","price","freight_value","statut_commande"]:
    assert  faits[col].isna().sum()==0, f"totale NULL dans {col}"
print("aucun NULL sur les colonnes obligatoire")

# region renseignees
assert faits["region_client_etat"].isna().sum()==0, f"region_client_etat NULL"
assert  faits["region_vendeur_etat"].isna().sum()==0, f"region_vendeur_etat NULL"
print("toutes les regions sont renseignees")

# appercu
print("appercu du dataframe finale:")
print(faits.head())

print("statistiques:")
print("totale observation :",len(faits))
print("total comandes distinctes :", faits["order_id"].nunique())
print("total produit distinct :",faits["produit_id"].nunique())
print("total vendeur distinct :", faits["vendeur_id"].nunique())
print("delai livraison moyen :", faits["delai_livraison_jour"].mean().round(3))
print("retard moyen :",faits["retard_jour"].mean().round(3))
print("note moyen :",faits["review_score"].mean().round(3))
print("chiffre d'affaire total: ", faits["price"].sum().round(3),"R$")

aucun doublon sur order_id et order_items_id
aucun NULL sur les colonnes obligatoire
toutes les regions sont renseignees
appercu du dataframe finale:
                           order_id  order_item_id  \
0  00010242fe8c5a6d1ba2dd792cb16214              1   
1  00018f77f2f0320c557190d7a144bdd3              1   
2  000229ec398224ef6ca0657da4fc703e              1   
3  00024acbcdf0a6daa1e931b038114c75              1   
4  00042b26cf59d7ce69dfabb4e55b4fd9              1   

                          client_id                        produit_id  \
0  3ce436f183e68e07877b285a838db11a  4244733e06e7ecb4970a6e2683c13e61   
1  f6dd3ec061db4e3987629fe6b26e5cce  e5f2d52b802189ee658865ca93d83a8f   
2  6489ae5e4333f3693df5ad4372dab6d3  c777355d18b72b67abbeef9df44fd0fd   
3  d4eb9395c8c0431ee92fce09860c5a06  7634da152a4610f1595efa32f14722fc   
4  58dbd0b2d70206bf40e62cd34e84d795  ac6c3623068f30de03045865e4e10089   

                         vendeur_id region_client_etat region_vendeur_etat  \
0  48436